## Processing S-BIAD1043 Fig-3 Conversion

### Imports

In [1]:
from lxml import etree
from io import StringIO, BytesIO
from os.path import join, split, isdir
from os import getcwd
from glob import glob
import uuid
from datetime import datetime
import tifffile
import numpy as np
from collections import defaultdict

from ome_types.model import Instrument, Microscope, Objective, InstrumentRef, OME, Image, Pixels, Channel, TiffData, Detector, Microscope_Type
from ome_types.model import Well, WellSample, WellSampleRef, Plate, ImageRef
from ome_types import from_xml, to_xml

### Paths and Variables

In [6]:
file_path="/Users/tushar/BIA/hackathon/fig_3_files/Files"
root_dir=file_path
manufacturer="Yokogawa"
uscope_model="CellVoyager CV7000/CV8000"
uscope_serial=""

### Metadata about the Instrument

In [7]:
uscope = Microscope(manufacturer=manufacturer, model=uscope_model, serial_number=uscope_serial)  # Read from metadata in a InLens image  (open image in notepad, search "crossbeam")
instrument = Instrument(id="Instrument:1", microscope=uscope)

In [13]:
img_name = glob(join(file_path, "*.tif"))[0]

### Functions to extract channels and field index

In [10]:
def extract_channels_from_image_name(img_name):
    meta = img_name.split("_")[-1].split(".tif")[0]
    return int(meta.split("C")[-1])

def extract_field_from_image_name(img_name):
    meta = img_name.split("_")[-1].split(".tif")[0]
    return int(meta.split("F")[-1].split("L")[0])
    

In [15]:
img_name.split("_")[-1]

'T0001F009L01A02Z01C04.tif'

In [ ]:
extract_channels_from_image_name(img_name)

In [ ]:
extract_field_from_image_name(img_name)

### Generating a Map with Image and its position in the well

In [ ]:
channel_dict = {1: "DAPI", 3: "DY-549P1", 4: "DY-647P1"}
channel_map = {1: 0, 3: 1, 4: 2}
well_field_d = defaultdict(lambda: defaultdict(dict))
for z, img_name in enumerate(glob(join(file_path, "*.tif"))):
    img_uuid = TiffData.UUID(file_name=f"{split(img_name)[1]}", value=f"urn:uuid:{uuid.uuid4()}")
    channel = extract_channels_from_image_name(img_name)
    well = img_name.split("_")[-2]
    field_index = extract_field_from_image_name(img_name)
    well_field_d[well][field_index][channel_map[channel]] = TiffData(uuid=img_uuid, ifd=0, first_c=channel_map[channel])

### Inspecting one image to read the metadata

In [ ]:
root = None
with tifffile.TiffFile(img_name) as tif:
    # Loop through all pages in the TIFF
    for page in tif.pages:
        for tag in page.tags.values():
            tag_name = tag.name
            tag_value = tag.value
            # Check if the tag contains XML
            if tag_name.__contains__("Strip"):
                continue
            print(f"{tag_name}: {tag_value}")

### Setting values for the pixels

In [ ]:
size_x=1998
size_y=1998
px_unit="µm"
x_px=0.028125
y_px=0.028125
z_px=1
dtype="uint16"

### Generating wells and Plate dict for writing to the companion file

In [ ]:
count=0
image_l = []
well_l = []
c0_o = Channel(id="Channel:1", name="DAPI")
c1_o = Channel(id="Channel:2", name="DY-549P1")
c2_o = Channel(id="Channel:3", name="DY-647P1")
well_map = {"E": 4, "F":5, "G":6}
for i, (well, field_d) in enumerate(well_field_d.items()):
    well_samples = []
    for field, channel_d in field_d.items():    
        c0 = channel_d[0]
        c1 = channel_d[1]
        c2 = channel_d[2]
        pixel_o = Pixels(id=f"Pixels:{count}", 
                         channels=[c0_o, c1_o ,c2_o], dimension_order="XYCZT",
                         size_x=size_x, size_y=size_y, size_z=1, size_c=3, size_t=1, type=dtype, physical_size_x=x_px, physical_size_x_unit=px_unit,
                        physical_size_y=y_px, physical_size_y_unit=px_unit, physical_size_z=z_px, physical_size_z_unit=px_unit,tiff_data_blocks=[c0,c1,c2]
                        )
        image_l.append(Image(id=f"Image:{count}", pixels=pixel_o))
        well_samples.append(WellSample(id=f"WellSample:{count}", image_ref=ImageRef(id=f"Image:{count}"), index=field))
        count+=1
    well_l.append(Well(id=f"Well:{i}", well_samples=well_samples, row=well_map[well[0]], column=well[1:]))
plate = Plate(id="Plate:1", wells=well_l,)

In [ ]:
well_field_d.keys()

In [ ]:
plate

### Writing the companion file

In [ ]:
image_name="PlatWell_fig_3"
ome_meta = OME(uuid=f"urn:uuid:{uuid.uuid4()}")
ome_meta.images.extend(image_l)
ome_meta.plates.append(plate)
ome_meta.instruments.append(instrument)

ome_str = to_xml(ome_meta) #, validate=True
with open(join(root_dir, image_name+".companion.ome"), "w", encoding="utf-8") as f:
    f.write(ome_str)